In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as alt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
# Load dataset
df=pd.read_csv(r'C:\Users\VAISHNAVI\Desktop\Internship project1\balanced_ai_human_prompts.csv')
df

,text,generated
0,"Machine learning, a subset of artificial intel...",1
1,"A decision tree, a prominent machine learning ...",1
2,"Education, a cornerstone of societal progress,...",1
3,"Computers, the backbone of modern technology, ...",1
4,"Chess, a timeless game of strategy and intelle...",1
...,...,...
2745,Generate a detailed summary of global healthca...,1
2746,Compose an in-depth exploration of financial t...,1
2747,Generate a detailed summary of autonomous vehi...,1
2748,Develop a persuasive argument about internet o...,1


In [3]:
# Copy dataset
dfc=df.copy()
dfc

,text,generated
0,"Machine learning, a subset of artificial intel...",1
1,"A decision tree, a prominent machine learning ...",1
2,"Education, a cornerstone of societal progress,...",1
3,"Computers, the backbone of modern technology, ...",1
4,"Chess, a timeless game of strategy and intelle...",1
...,...,...
2745,Generate a detailed summary of global healthca...,1
2746,Compose an in-depth exploration of financial t...,1
2747,Generate a detailed summary of autonomous vehi...,1
2748,Develop a persuasive argument about internet o...,1


In [4]:
dfc.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
2745    False
2746    False
2747    False
2748    False
2749    False
Length: 2750, dtype: bool

In [5]:
# data cleaning
dfc.drop_duplicates()

,text,generated
0,"Machine learning, a subset of artificial intel...",1
1,"A decision tree, a prominent machine learning ...",1
2,"Education, a cornerstone of societal progress,...",1
3,"Computers, the backbone of modern technology, ...",1
4,"Chess, a timeless game of strategy and intelle...",1
...,...,...
2745,Generate a detailed summary of global healthca...,1
2746,Compose an in-depth exploration of financial t...,1
2747,Generate a detailed summary of autonomous vehi...,1
2748,Develop a persuasive argument about internet o...,1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2750 entries, 0 to 2749
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       2750 non-null   object
 1   generated  2750 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 43.1+ KB


In [7]:
'keep' in stopwords.words('english')

False

In [8]:
# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Text cleaning function
def clean_text(text):
    text = text.lower()                           # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text)          # Remove punctuation, numbers
    tokens = text.split()                         # Tokenization
    tokens = [word for word in tokens if word not in stop_words]  # Stopword removal
    tokens = [lemmatizer.lemmatize(word) for word in tokens]      # Lemmatization
    return " ".join(tokens)

# Apply cleaning to dataset
dfc['clean_text'] = df['text'].apply(clean_text)

# View results
dfc[['text', 'clean_text']].head()


,text,clean_text
0,"Machine learning, a subset of artificial intel...",machine learning subset artificial intelligenc...
1,"A decision tree, a prominent machine learning ...",decision tree prominent machine learning algor...
2,"Education, a cornerstone of societal progress,...",education cornerstone societal progress extend...
3,"Computers, the backbone of modern technology, ...",computer backbone modern technology revolution...
4,"Chess, a timeless game of strategy and intelle...",chess timeless game strategy intellect transce...


In [9]:
x=dfc["clean_text"]
x

0       machine learning subset artificial intelligenc...
1       decision tree prominent machine learning algor...
2       education cornerstone societal progress extend...
3       computer backbone modern technology revolution...
4       chess timeless game strategy intellect transce...
                              ...                        
2745    generate detailed summary global healthcare co...
2746    compose indepth exploration financial technolo...
2747    generate detailed summary autonomous vehicle c...
2748    develop persuasive argument internet thing con...
2749    generate detailed summary supply chain managem...
Name: clean_text, Length: 2750, dtype: object

In [10]:
y=dfc["generated"]
y

0       1
1       1
2       1
3       1
4       1
       ..
2745    1
2746    1
2747    1
2748    1
2749    1
Name: generated, Length: 2750, dtype: int64

In [11]:
# preprocessing
from sklearn.model_selection import train_test_split

In [12]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [13]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VAISHNAVI\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\VAISHNAVI\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [14]:
# Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1,2) )


x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)



In [15]:
x_train_vec.shape

(2200, 5000)

In [16]:
x_test_vec.shape

(550, 5000)

In [17]:
#model fitting
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train SVM on vectorized data
svm_model = LinearSVC()
svm_model.fit(x_train_vec, y_train)

# Predict on test data
y_pred_svm = svm_model.predict(x_test_vec)

# Evaluation
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))


SVM Accuracy: 0.9963636363636363

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00       284
           1       1.00      0.99      1.00       266

    accuracy                           1.00       550
   macro avg       1.00      1.00      1.00       550
weighted avg       1.00      1.00      1.00       550


Confusion Matrix:
 [[284   0]
 [  2 264]]


In [18]:
import joblib

joblib.dump(svm_model, "svm_text_detection_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']